In [ ]:
import pickle
import shutil
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.warp import reproject, transform_bounds
from rasterio.windows import from_bounds, Window
from rasterio.crs import CRS

# ==============================
# BG: Комбиниране на класификационни растерни плочки (10 m)
# EN: Merging classification raster tiles (10 m)
# ==============================

# ------------------------------
# BG: 1. Дефиниране на пътищата
# EN: 1. Define paths
# ------------------------------

# BG: Входна директория с резултатите от обектно-базирана класификация
# EN: Input directory with object-based classification results
input_dir = Path(r"C:\Users\steph\Downloads\master_thesis\exports\object_classification_results")

# BG: Изходна директория (подпапка "B_files_combined")
# EN: Output directory (subfolder "B_files_combined")
output_dir = input_dir / "B_files_combined"
output_dir.mkdir(exist_ok=True)

# BG: Път до изходния GeoTIFF файл
# EN: Path to the output GeoTIFF file
output_tif = output_dir / "combined_classification_10m.tif"

# BG: Референтен растер за целевата координатна система
# EN: Reference raster for the target CRS
ref_tif = Path(r"C:\Users\steph\Downloads\master_thesis\exports\sentinel2_fire_images\classification_report\random_forest\fire_04_random_forest_classification.tif")

# ------------------------------
# BG: 2. Извличане на целевата координатна система (CRS) от референтния файл
# EN: 2. Extract target CRS from the reference file
# ------------------------------
with rasterio.open(ref_tif) as ref:
    target_crs = ref.crs
    if target_crs is None:
        raise ValueError("BG: Референтният файл няма CRS! Задайте ръчно, напр. CRS.from_epsg(32633)\n"
                         "EN: Reference file has no CRS! Set manually, e.g. CRS.from_epsg(32633)")
# BG: Извеждане на информация за целевия CRS
# EN: Print target CRS info
print(f"Target CRS: {target_crs}")

# ------------------------------
# BG: 3. Събиране на входните TIF файлове и имената на класовете
# EN: 3. Collect input TIF files and class names
# ------------------------------
tif_paths = []
class_names = None

# BG: Обхождане на поддиректориите, започващи с "B"
# EN: Loop through subdirectories starting with "B"
for subdir in input_dir.glob("B*"):
    if not subdir.is_dir() or subdir == output_dir:
        continue
    # BG: Търсене на файлове, завършващи на "_obia_true.tif"
    # EN: Search for files ending with "_obia_true.tif"
    tif_files = list(subdir.glob("*_obia_true.tif"))
    if not tif_files:
        print(f"Warning: No TIF in {subdir}")
        continue
    tif_paths.append(tif_files[0])
    # BG: Ако имената на класовете още не са заредени, пробваме да ги вземем от .pkl файл
    # EN: If class names haven't been loaded yet, try to get them from a .pkl file
    if class_names is None:
        pkl_files = list(subdir.glob("*_class_names.pkl"))
        if pkl_files:
            with open(pkl_files[0], "rb") as f:
                class_names = pickle.load(f)
            print(f"Loaded classes from {pkl_files[0].name}")

if not tif_paths:
    raise RuntimeError("No TIF files found.")
print(f"\nFound {len(tif_paths)} tiles.")

# ------------------------------
# BG: 4. Обединени граници в целевата координатна система
# EN: 4. Union of all bounds in the target CRS
# ------------------------------
all_bounds = []
for path in tif_paths:
    with rasterio.open(path) as src:
        # BG: Ако източникът няма CRS, приемаме целевия
        # EN: If source has no CRS, assume target
        src_crs = src.crs if src.crs is not None else target_crs
        bounds = transform_bounds(src_crs, target_crs, *src.bounds)
        all_bounds.append(bounds)

# BG: Минимални и максимални стойности
# EN: Minimum and maximum values
union_left   = min(b[0] for b in all_bounds)
union_bottom = min(b[1] for b in all_bounds)
union_right  = max(b[2] for b in all_bounds)
union_top    = max(b[3] for b in all_bounds)

# BG: Размер на пиксела – 10 m
# EN: Pixel size – 10 m
res = 10.0

# BG: Размери на изходния растер (добавен малък буфер)
# EN: Output raster dimensions (with a small buffer)
output_width  = int(np.ceil((union_right - union_left) / res)) + 10
output_height = int(np.ceil((union_top - union_bottom) / res)) + 10

# BG: Трансформация от горен ляв ъгъл
# EN: Transform from top-left corner
out_transform = rasterio.transform.from_origin(union_left, union_top, res, res)

print(f"Output size: {output_width} x {output_height}")
print(f"Output bounds: left={union_left}, bottom={union_bottom}, right={union_right}, top={union_top}")

# ------------------------------
# BG: 5. Създаване на празен изходен GeoTIFF
# EN: 5. Create empty output GeoTIFF
# ------------------------------
out_meta = {
    "driver": "GTiff",
    "height": output_height,
    "width": output_width,
    "count": 1,                           # BG: един канал / EN: single band
    "dtype": "uint8",
    "crs": target_crs,
    "transform": out_transform,
    "compress": "lzw",
    "nodata": 0,
    "BIGTIFF": "YES",                     # BG: голям файл / EN: large file
    "tiled": True,
    "blockxsize": 512,                    # BG: размер на плочката / EN: tile size
    "blockysize": 512,
}
with rasterio.open(output_tif, "w", **out_meta) as dst:
    pass
print("Empty output file created.\n")

# ------------------------------
# BG: 6. Препроектиране и вмъкване на всяка плочка в мозайката
# EN: 6. Reproject and insert each tile into the mosaic
# ------------------------------
for i, path in enumerate(tif_paths, start=1):
    print(f"Processing tile {i}/{len(tif_paths)}: {path.name}")
    with rasterio.open(path) as src:
        # BG: Използване на CRS от файла или целевия
        # EN: Use CRS from file or target
        src_crs = src.crs if src.crs is not None else target_crs
        if src.crs is None:
            print("  (CRS was None, using target CRS)")

        data = src.read(1)                # BG: първи канал / EN: first band
        src_transform = src.transform

        # BG: Ако трансформацията е "north‑down", обръщаме изображението вертикално
        # EN: If transform is north‑down, flip the image vertically
        if src_transform.e > 0:
            print("  Flipping north‑down tile")
            data = np.flipud(data)
            src_transform = rasterio.Affine(
                src_transform.a, src_transform.b, src_transform.c,
                src_transform.d, -src_transform.e,
                src_transform.f + src.height * src_transform.e
            )

        # BG: Граници на източника в собствената му CRS
        # EN: Source bounds in its own CRS
        src_bounds = rasterio.transform.array_bounds(src.height, src.width, src_transform)
        # BG: Трансформиране на границите в целевата CRS
        # EN: Transform bounds to target CRS
        target_bounds = transform_bounds(src_crs, target_crs, *src_bounds)

        # BG: Прозорец в изходните пикселни координати
        # EN: Window in output pixel coordinates
        window = from_bounds(*target_bounds, transform=out_transform)
        window = window.round_offsets(op='floor', pixel_precision=0)
        window = window.round_lengths(op='ceil', pixel_precision=0)

        # BG: Отваряне на изходния файл за четене и запис
        # EN: Open output file for read/write
        with rasterio.open(output_tif, "r+") as dst:
            out_window = Window(0, 0, dst.width, dst.height)
            try:
                # BG: Пресичане на изчисления прозорец с границите на изходния растер
                # EN: Intersect computed window with output raster extent
                window = window.intersection(out_window)
            except rasterio.errors.WindowError:
                print(f"  Warning: {path.name} outside output bounds – skipped.")
                continue

            if window.width <= 0 or window.height <= 0:
                print(f"  Tile outside output bounds – skipped.")
                continue

            # BG: Празен масив за изрязаната част
            # EN: Empty array for the window region
            dst_array = np.zeros((window.height, window.width), dtype=data.dtype)
            dst_transform = rasterio.windows.transform(window, out_transform)

            # BG: Препроектиране на данните
            # EN: Reproject the data
            reproject(
                source=data,
                destination=dst_array,
                src_transform=src_transform,
                src_crs=src_crs,
                dst_transform=dst_transform,
                dst_crs=target_crs,
                resampling=Resampling.nearest,    # BG: най-близък съсед / EN: nearest neighbour
            )

            # BG: Записване в изходния растер
            # EN: Write to output raster
            dst.write(dst_array, 1, window=window)
            print(f"  Written window {window}")

print(f"\nMerged raster saved to: {output_tif}")

# ------------------------------
# BG: 7. Примерна проверка на валидността
# EN: 7. Validation sample
# ------------------------------
with rasterio.open(output_tif) as dst:
    sample = dst.read(1, window=Window(0,0,100,100))
    print(f"Sample min: {sample.min()}, max: {sample.max()}")

# BG: Почистване на временната директория
# EN: Clean up temporary directory
temp_dir = tempfile.mkdtemp()
shutil.rmtree(temp_dir, ignore_errors=True)

# ------------------------------
# BG: 8. Визуализация (с намалена разделителна способност)
# EN: 8. Visualise (down‑sampled)
# ------------------------------
with rasterio.open(output_tif) as dst:
    # BG: Коефициент на намаляване
    # EN: Downsampling factor
    downsample = 100
    ds_height = dst.height // downsample
    ds_width  = dst.width // downsample
    data = dst.read(1, out_shape=(ds_height, ds_width), resampling=Resampling.nearest)

fig, ax = plt.subplots(figsize=(12, 10))
# BG: Показване с цветова карта tab20
# EN: Display using tab20 colormap
im = ax.imshow(data, cmap="tab20", interpolation="none")

# BG: Добавяне на легенда с имената на класовете (ако са налични)
# EN: Add legend with class names (if available)
if class_names:
    unique_classes = np.unique(data)
    unique_classes = unique_classes[unique_classes != 0]  # BG: изключва "некласифициран" / EN: exclude "unclassified"
    colors = [im.cmap(im.norm(cls)) for cls in unique_classes]
    patches = [mpatches.Patch(color=colors[i],
                              label=f"{cls}: {class_names.get(cls, 'unknown')}")
               for i, cls in enumerate(unique_classes)]
    ax.legend(handles=patches, bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()

# BG: Заглавие на графиката
# EN: Plot title
ax.set_title(f"Merged Classification (10 m) – down‑sampled {downsample}×")
ax.set_xlabel("Column (pixel)")
ax.set_ylabel("Row (pixel)")
plt.show()